# L-1 Step 1：規則快篩

以關鍵詞 + 句型規則對全量日報進行高速初步分類，免費完成 50–60% 的 L 層標注。

| 子任務 | 規格 |
|--------|------|
| A 規則詞典建立 | L1–L7 關鍵詞集合 + 否定詞，補入問卷多選選項高頻標準答案 |
| B 短文本特例 | < 50 字：命中→直接標注；無命中→uncertain，不進 Step 2 |
| C 信心路由 | >=3 強關鍵詞->HIGH（直接定案）；1-2->MEDIUM（進 Step 2）；0->進 Step 2 |
| D 全量快篩 | 讀取全量日報，套用規則，寫出分類結果 JSONL |
| E 覆蓋率報告 | 各層覆蓋率 + HIGH/MEDIUM/SKIP 分布，驗證 50-60% 目標 |

| 完成標準 | 目標 |
|----------|------|
| HIGH 覆蓋率 | >= 50% |
| HIGH 信心抽查正確率 | >= 95% |


## 0. 安裝套件

In [1]:
!pip install pyodbc pandas tqdm -q

## 1. 匯入套件 & 全域設定

In [2]:
import re, json, time, hashlib, configparser
from pathlib import Path
from datetime import datetime

import pyodbc
import pandas as pd
from tqdm import tqdm

CFG_SQL    = r'D:\yujui\SqlServer18.txt'
OUTPUT_DIR = Path(r'D:\yujui\痛點需求地圖\step1_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DICT_PATH    = OUTPUT_DIR / 'l_layer_rules.json'
RESULT_JSONL = OUTPUT_DIR / 'step1_results.jsonl'
COVERAGE_CSV = OUTPUT_DIR / 'step1_coverage.csv'

HIGH_THRESHOLD   = 2    # 命中 >= N 個強關鍵詞 -> HIGH（從 3 降為 2，覆蓋率 18%->21%）
MEDIUM_THRESHOLD = 1    # 命中 >= N 個 -> MEDIUM
SHORT_TEXT_LEN   = 50   # < N 字為短文本
SQL_DATABASE     = 'DSC_CRM_SP'

print(f'輸出目錄：{OUTPUT_DIR}')


輸出目錄：D:\yujui\痛點需求地圖\step1_output


## 2. 建立 SQL Server 連線

In [3]:
cfg = configparser.ConfigParser()
cfg.read(CFG_SQL, encoding='utf-8')
if 'mssql' not in cfg:
    raise RuntimeError(f'找不到 [mssql] 區段：{CFG_SQL}')
sec = cfg['mssql']

conn = pyodbc.connect(
    f"DRIVER={{SQL Server}};SERVER={sec['server']};DATABASE={SQL_DATABASE};"
    f"UID={sec['uid']};PWD={sec['pwd']};",
    autocommit=True
)
print(f'SQL Server 連線成功：{sec["server"]}')


SQL Server 連線成功：10.20.99.18


---
## 子任務 A：規則詞典建立
各 L 層關鍵詞集合 + 否定詞。v4.2 更新：問卷多選選項高頻標準答案補入對應 L 層。


In [4]:
# ── A1：定義 L 層規則詞典 ──────────────────────────────────
# 格式：{ l_layer: { keywords: [...], negations: [...] } }
# keywords：強關鍵詞（命中計數用）
# negations：否定詞（命中則排除該層）

L_RULES = {
    'L1': {
        'keywords': [
            '阻礙', '卡關', '無法推進', '第一次', '困難', '問題', '障礙', '痛點',
            '導入失敗', '失敗原因', '無法使用', '系統錯誤', '無法操作',
            # 問卷多選選項補充
            '更高的存貨呆滯風險', '資金周轉困難', '人力不足', '技術門檻高',
        ],
        'negations': ['已解決', '順利', '完成'],
    },
    'L2': {
        'keywords': [
            '老闆', '主管', 'KPI', '績效', '部門', '決策', '長官', '董事長',
            '壓力', '考核', '目標達成', '業績壓力', '上級', '主席',
            # 問卷多選選項補充
            '上級顧慮多', '上級顧慮多，難以獲准投資計畫', '主管要求', '高層指示',
        ],
        'negations': [],
    },
    'L3': {
        'keywords': [
            '轉型', '目標', '策略', '計畫', '三年', '未來', '佈局', '戰略',
            '數位化', '智慧化', '整體規劃', '藍圖', '長期',
            # 問卷多選選項補充
            '更嚴苛的成本競爭', '更短的交期要求', '數位轉型壓力', '產業升級',
        ],
        'negations': [],
    },
    'L4': {
        'keywords': [
            '產業', '趨勢', '法規', '政策', '競爭', 'ESG', '供應鏈',
            '市場', '同業', '競爭對手', '產業鏈', '外部環境', '碳排',
            # 問卷多選選項補充
            '法規遵循', '環保要求', '國際標準', '供應鏈風險',
        ],
        'negations': [],
    },
    'L5': {
        'keywords': [
            '評估', '比較', '需求確認', '想了解', '能不能', '有沒有',
            '功能', '模組', '報價', 'Demo', '展示', 'POC', '試用',
            # 問卷多選選項補充
            '功能是否符合', '系統整合', '客製化需求',
        ],
        'negations': [],
    },
    'L6': {
        'keywords': [
            '態度', '溫度', '變化', '猶豫', '轉變', '比上次',
            '更積極', '更保守', '態度轉', '轉正', '轉冷', '觀望',
            # 問卷多選選項補充
            '決策態度改變', '意願轉變',
        ],
        'negations': [],
    },
    'L7': {
        'keywords': [
            '合約', '簽訂', '成交', '付款', '方案確認', '折扣',
            '簽約', '下單', '訂單', '結案', 'PO', '採購單', '收款',
            # 問卷多選選項補充
            '合約談判', '付款條件', '驗收條件',
        ],
        'negations': ['取消', '退訂'],
    },
}

with open(DICT_PATH, 'w', encoding='utf-8') as f:
    json.dump(L_RULES, f, ensure_ascii=False, indent=2)

print(f'規則詞典已存：{DICT_PATH}')
for layer, rules in L_RULES.items():
    print(f'  {layer}：{len(rules["keywords"])} 個關鍵詞，{len(rules["negations"])} 個否定詞')


規則詞典已存：D:\yujui\痛點需求地圖\step1_output\l_layer_rules.json
  L1：17 個關鍵詞，3 個否定詞
  L2：18 個關鍵詞，0 個否定詞
  L3：17 個關鍵詞，0 個否定詞
  L4：17 個關鍵詞，0 個否定詞
  L5：16 個關鍵詞，0 個否定詞
  L6：14 個關鍵詞，0 個否定詞
  L7：16 個關鍵詞，2 個否定詞


---
## 子任務 B+C：快篩函式（短文本特例 + 信心路由）


In [5]:
def classify_rule(text: str, rules: dict) -> dict:
    """
    套用規則快篩。
    回傳：{ l_layer, confidence(HIGH/MEDIUM/SKIP), hit_count, hit_keywords, is_short_text }
    """
    is_short = len(text) < SHORT_TEXT_LEN
    best_layer, best_hits, best_keywords = None, 0, []

    for layer, cfg in rules.items():
        if any(neg in text for neg in cfg['negations']):
            continue
        hits = [kw for kw in cfg['keywords'] if kw in text]
        if len(hits) > best_hits:
            best_hits, best_layer, best_keywords = len(hits), layer, hits

    # 短文本特例
    if is_short:
        if best_hits >= 1:
            confidence = 'HIGH'
        else:
            return {'l_layer': 'uncertain', 'confidence': 'SKIP',
                    'hit_count': 0, 'hit_keywords': [], 'is_short_text': True}
    else:
        if best_hits >= HIGH_THRESHOLD:
            confidence = 'HIGH'
        elif best_hits >= MEDIUM_THRESHOLD:
            confidence = 'MEDIUM'
        else:
            confidence = 'MEDIUM'  # 0 命中也進 Step 2
            best_layer = None

    return {
        'l_layer':       best_layer or 'uncertain',
        'confidence':    confidence,
        'hit_count':     best_hits,
        'hit_keywords':  best_keywords,
        'is_short_text': is_short,
    }

# 單筆測試
test_cases = [
    '老闆要求今年 KPI 達成，主管施加很大的業績壓力，部門目標很緊',
    '客戶問有沒有 ERP 模組可以 Demo，想了解庫存功能',
    '簽約',
    '今天拜訪客戶',
]
for t in test_cases:
    r = classify_rule(t, L_RULES)
    print(f"[{r['l_layer']:>10}][{r['confidence']:>6}] 命中{r['hit_count']}個 | {t[:40]}")


[        L2][  HIGH] 命中6個 | 老闆要求今年 KPI 達成，主管施加很大的業績壓力，部門目標很緊
[        L5][  HIGH] 命中5個 | 客戶問有沒有 ERP 模組可以 Demo，想了解庫存功能
[        L7][  HIGH] 命中1個 | 簽約
[ uncertain][  SKIP] 命中0個 | 今天拜訪客戶


---
## 子任務 D：全量日報快篩
從 SQL Server 讀取全量日報，套用規則，結果寫出 `step1_results.jsonl`。


In [6]:
# ── D1：讀取全量日報 ──────────────────────────────────────
query = '''
SELECT
    GY001 AS company_id,
    GY003 AS salesperson_id,
    GY004 AS contact_date,
    GY012 AS log_content,
    LEN(GY012) AS log_len,
    FORMAT(CREATE_DATE,'yyyy-MM') AS ym
FROM DSC_CRM_SP..CRMGY
WHERE LEN(LTRIM(RTRIM(GY012))) >= 5
  AND GY012 NOT LIKE '%MA簽回%'
  AND GY012 NOT LIKE '%MA簽約%'
  AND GY012 NOT LIKE '%維護合約%'
  AND GY012 NOT LIKE '%廢止%'
'''

print('讀取全量日報中...')
logs_df = pd.read_sql(query, conn)
logs_df['log_content'] = logs_df['log_content'].astype(str).str.replace('\n', ' ', regex=False)
logs_df['event_id'] = logs_df['log_content'].apply(
    lambda t: __import__('hashlib').sha256(t.encode()).hexdigest()[:16]
)
print(f'全量日報：{len(logs_df):,} 筆')
print(f'短文本（< {SHORT_TEXT_LEN} 字）：{(logs_df["log_len"] < SHORT_TEXT_LEN).sum():,} 筆')


讀取全量日報中...


C:\Users\DSC\AppData\Local\Temp\ipykernel_71360\3692217870.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  logs_df = pd.read_sql(query, conn)


全量日報：4,141,953 筆
短文本（< 50 字）：3,037,919 筆


In [7]:
# ── D2：套用規則快篩，斷點續跑，寫出 step1_results.jsonl ──
RESUME = True

done_ids = set()
if RESUME and RESULT_JSONL.exists():
    done_ids = {json.loads(l)['event_id'] for l in open(RESULT_JSONL, encoding='utf-8') if l.strip()}
    print(f'已處理：{len(done_ids):,} 筆，繼續從斷點執行')

todo = logs_df[~logs_df['event_id'].isin(done_ids)]
print(f'待處理：{len(todo):,} 筆')

stats = {'HIGH': 0, 'MEDIUM': 0, 'SKIP': 0}

with open(RESULT_JSONL, 'a', encoding='utf-8') as f:
    for _, row in tqdm(todo.iterrows(), total=len(todo), desc='規則快篩'):
        result = classify_rule(row['log_content'], L_RULES)
        stats[result['confidence']] = stats.get(result['confidence'], 0) + 1
        record = {
            'event_id':         row['event_id'],
            'company_id':       row['company_id'],
            'ym':               row['ym'],
            'log_len':          int(row['log_len']),
            'step1_result':     result['l_layer'],
            'step1_confidence': result['confidence'],
            'hit_count':        result['hit_count'],
            'hit_keywords':     result['hit_keywords'],
            'is_short_text':    result['is_short_text'],
            'classified_at':    datetime.now().isoformat(),
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

total = sum(stats.values()) + len(done_ids)
print(f'\n快篩完成：{total:,} 筆')
for k in ['HIGH', 'MEDIUM', 'SKIP']:
    print(f'  {k}：{stats[k]:,} 筆')


已處理：3,537,579 筆，繼續從斷點執行
待處理：12 筆


規則快篩: 100%|██████████| 12/12 [00:00<?, ?it/s]


快篩完成：3,537,591 筆
  HIGH：6 筆
  MEDIUM：1 筆
  SKIP：5 筆


---
## 子任務 E：覆蓋率報告
驗證 HIGH 覆蓋率 >= 50%，以及各層分布是否均衡。若未達標需補充 L_RULES 詞典後重跑 Cell D2。


In [8]:
results = [json.loads(l) for l in open(RESULT_JSONL, encoding='utf-8') if l.strip()]
df = pd.DataFrame(results)

total    = len(df)
high_n   = (df['step1_confidence'] == 'HIGH').sum()
medium_n = (df['step1_confidence'] == 'MEDIUM').sum()
skip_n   = (df['step1_confidence'] == 'SKIP').sum()
coverage = high_n / total * 100

print(f'總筆數               ：{total:,}')
print(f'HIGH  （直接定案）  ：{high_n:,}  {coverage:.1f}%  {"✅" if coverage >= 50 else "❌ 需補充詞典"}')
print(f'MEDIUM（進 Step 2） ：{medium_n:,}  {medium_n/total*100:.1f}%')
print(f'SKIP  （短文本）    ：{skip_n:,}  {skip_n/total*100:.1f}%')

high_df    = df[df['step1_confidence'] == 'HIGH']
layer_dist = high_df['step1_result'].value_counts().sort_index()
print('\n【HIGH 各層分布】')
for layer in ['L1','L2','L3','L4','L5','L6','L7','uncertain']:
    n = int(layer_dist.get(layer, 0))
    pct = n / high_n * 100 if high_n > 0 else 0
    print(f'  {layer}：{n:>6,} 筆  {pct:.1f}%')


總筆數               ：4,141,953
HIGH  （直接定案）  ：867,023  20.9%  ❌ 需補充詞典
MEDIUM（進 Step 2） ：961,849  23.2%
SKIP  （短文本）    ：2,313,081  55.8%

【HIGH 各層分布】
  L1：118,076 筆  13.6%
  L2：190,008 筆  21.9%
  L3：68,938 筆  8.0%
  L4：30,981 筆  3.6%
  L5：314,471 筆  36.3%
  L6： 6,667 筆  0.8%
  L7：137,882 筆  15.9%
  uncertain：     0 筆  0.0%


In [9]:
report_rows = []
for layer in ['L1','L2','L3','L4','L5','L6','L7','uncertain']:
    n = int(layer_dist.get(layer, 0))
    report_rows.append({
        'l_layer':   layer,
        'high_count': n,
        'high_pct':  round(n / high_n * 100, 2) if high_n > 0 else 0,
    })
report_rows.append({'l_layer': 'TOTAL_HIGH',   'high_count': int(high_n),   'high_pct': round(coverage, 2)})
report_rows.append({'l_layer': 'TOTAL_MEDIUM', 'high_count': int(medium_n), 'high_pct': round(medium_n/total*100, 2)})
report_rows.append({'l_layer': 'TOTAL_SKIP',   'high_count': int(skip_n),   'high_pct': round(skip_n/total*100, 2)})

pd.DataFrame(report_rows).to_csv(COVERAGE_CSV, index=False, encoding='utf-8-sig')

print('=' * 55)
print('Step 1 完成')
print(f'  step1_results.jsonl : {RESULT_JSONL}')
print(f'  step1_coverage.csv  : {COVERAGE_CSV}')
print(f'  l_layer_rules.json  : {DICT_PATH}')
status = '✅' if coverage >= 50 else '❌ 需補充詞典後重跑 Cell D2'
print(f'  HIGH 覆蓋率：{coverage:.1f}%  {status}')


Step 1 完成
  step1_results.jsonl : D:\yujui\痛點需求地圖\step1_output\step1_results.jsonl
  step1_coverage.csv  : D:\yujui\痛點需求地圖\step1_output\step1_coverage.csv
  l_layer_rules.json  : D:\yujui\痛點需求地圖\step1_output\l_layer_rules.json
  HIGH 覆蓋率：20.9%  ❌ 需補充詞典後重跑 Cell D2
